# Notebook 04 — TinyViT on MNIST even/odd

**Architecture:** Single-block TinyViT (embed_dim=32, n_heads=2, ffn_dim=64) trained on MNIST even/odd (all 10 digits).

**BFT layers (forward order):** B0-V (attn, W_V) → B0-O (fc, W_O) → B0-FFN1 (fc) → B0-FFN2 (fc)

## Structure
0. **Imports & setup**
1. **Configuration**
2. **Backward Factor Trace** — seed-0 model, BFT, exploratory plots (factor panels, attention grounding, factor tree)
3. **BFT figures** — main paper and appendix (placeholders)
4. **Fingerprints** — NNLS round-trip, near-OOD (Fashion-MNIST), far-OOD, fingerprint similarity, embeddings
5. **Fingerprint figures** — main paper and appendix (placeholders)

Validation, robustness and ablation analyses live in notebook 09.

## §0 — Imports & setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
import torchvision.transforms as T
from sklearn.metrics.pairwise import cosine_similarity, paired_cosine_distances
from torch.utils.data import DataLoader, TensorDataset

from src import (
    TinyViT, get_mnist_loaders, label_transform_even_odd, bft, extract_tree_nodes,
    plot_factor_tree, compute_node_activations, extract_fingerprint_matrix,
    project_stimuli_onto_tree, save_experiment, load_experiment, plot_factor_overview_panel,
    plot_factor_gallery, plot_input_layer_factors, plot_embedding_comparison,
)

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 80})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

## §1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_ROOT = '../data/models'
FIG_DIR    = '../figs/04_vit'
MODEL_DIR  = os.path.join(MODEL_ROOT, 'mnist_even_odd_vit_tiny_seed0')
os.makedirs(FIG_DIR, exist_ok=True)

# ── Model hyperparameters ─────────────────────────────────────────────────────
EMBED_DIM   = 32
N_HEADS     = 2
FFN_DIM     = 64
N_CLASSES   = 2
CLASS_NAMES = {0: 'even', 1: 'odd'}
IMAGE_SIDE  = 28
N_EPOCHS    = 30

# ── BFT hyperparameters ───────────────────────────────────────────────────────
# k_max and n_branches in FORWARD layer order: [B0-V, B0-O, B0-FFN1, B0-FFN2]
# PUBLICATION SETTINGS (see PUBLICATION_SETTINGS.md). k_max from the nb09 S10 sweep
# (`rank x0.7`, run 4): silhouette 0.150 -> 0.203, NMF stability 0.830 -> 0.907,
# fingerprint 154 -> 75 dims. NOTE: layer-dict mode has no causal reconstruction, so this
# selection is constrained by clustering and stability only -- not by faithfulness.
K_MAX          = [7, 10, 7, 10]   # C0 circuit tree (nb15 vit, k_cap 16)
N_BRANCHES     = [1, 1, 2, 10]   # C0 circuit tree (nb15)
STIM_THRESHOLD = 0.0

# ── Fingerprint / OOD config (§4) ─────────────────────────────────────────────
N_TOP_GALLERY = 6
N_FAR         = 500
N_FP          = 100

BASE_CONFIG = {
    'arch': 'TinyViT',
    'arch_kwargs': {
        'embed_dim': EMBED_DIM, 'n_heads': N_HEADS,
        'ffn_dim': FFN_DIM, 'n_classes': N_CLASSES,
    },
    'dataset': 'MNIST',
    'dataset_kwargs': {'root': '../data/', 'batch_size': 64},
    'label_transform': 'even_odd',
}
LABEL_TRANSFORM = label_transform_even_odd
print('Config OK')
print(f'MODEL_DIR: {MODEL_DIR}')

# ── Fingerprint-tree HPs (paper/submission settings) — §4/§5 only ─────────────
K_MAX_FP     = [6, 4, 4, 3]
N_BRANCHES_FP = [1, 1, 2, 4]


## §2 — Backward Factor Trace (seed 0)

BFT traces through all 4 layers of the transformer block in reverse order:
```
B0-FFN2 (root) → B0-FFN1 → B0-O → B0-V (attn, leaf)
```
The attention layer uses an **attention-weighted effective input**:
$$x^\text{eff}_n = \sum_{j=0}^{T-1} \alpha_{n,j}\, x_{n,j}$$
so each sample is represented by one blended token vector.

### 2a — Data, model and layer activations

In [ ]:
train_loader, test_loader = get_mnist_loaders(batch_size=64, root='../data/')
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ── Seed-0 model: load or train ───────────────────────────────────────────────
def train_vit(seed_dir, n_epochs=N_EPOCHS, seed=0):
    """Train TinyViT from scratch with Adam, save checkpoint."""
    torch.manual_seed(seed)
    m = TinyViT(embed_dim=EMBED_DIM, n_heads=N_HEADS, ffn_dim=FFN_DIM,
                n_classes=N_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.NLLLoss()

    for epoch in range(n_epochs):
        m.train()
        total_loss = 0.0
        for imgs, digits in train_loader:
            imgs   = imgs.to(DEVICE)
            labels = LABEL_TRANSFORM(digits).to(DEVICE)
            loss   = criterion(m(imgs), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0 or epoch == n_epochs - 1:
            print(f'  epoch {epoch+1:3d}/{n_epochs}  loss={total_loss/len(train_loader):.4f}')

    cfg = dict(BASE_CONFIG, description='TinyViT on MNIST even/odd (seed 0)')
    save_experiment(m, cfg, seed_dir)
    print(f'  → saved to {seed_dir}')
    return m


if os.path.exists(os.path.join(MODEL_DIR, 'weights.pt')):
    model0, _ = load_experiment(MODEL_DIR, device=DEVICE)
    print(f'Loaded model from {MODEL_DIR}')
else:
    print('No checkpoint found, training seed 0 …')
    model0 = train_vit(MODEL_DIR)

model0.eval()
n_correct = 0
with torch.no_grad():
    for imgs, digits in test_loader:
        imgs   = imgs.to(DEVICE)
        labels = LABEL_TRANSFORM(digits).to(DEVICE)
        n_correct += (model0(imgs).argmax(1) == labels).sum().item()
print(f'Seed-0 test acc = {n_correct / len(test_loader.dataset):.4f}')

In [ ]:
def collect_vit_activations(model, loader, device):
    """Collect layer activations and images for correctly-classified samples."""
    model.eval()
    acts = {'attn_in': [], 'attn_w': [], 'attn_out_cls': [], 'ffn1_in': [], 'ffn2_in': []}
    images, targets, digits_all = [], [], []

    with torch.no_grad():
        for imgs, digs in loader:
            imgs   = imgs.to(device)
            labels = label_transform_even_odd(digs).to(device)
            logits = model(imgs, capture=True)
            mask   = (logits.argmax(1) == labels)
            if not mask.any():
                continue
            blk = model.block
            acts['attn_in'].append(blk._attn_in[mask].cpu().numpy())           # (B, 17, 32)
            acts['attn_w'].append(blk._attn_w[mask].mean(1)[:, 0, :].cpu().numpy())  # (B, 17) CLS row
            acts['attn_out_cls'].append(blk._attn_out[mask, 0].cpu().numpy())   # (B, 32)
            acts['ffn1_in'].append(blk._ffn1_in[mask, 0].cpu().numpy())         # (B, 32)
            acts['ffn2_in'].append(blk._ffn2_in[mask, 0].cpu().numpy())         # (B, 64)
            images.append(imgs[mask].cpu().numpy())
            targets.append(labels[mask].cpu().numpy())
            digits_all.append(digs[mask.cpu()].numpy())

    for k in acts:
        acts[k] = np.concatenate(acts[k], axis=0)
    images  = np.concatenate(images,  axis=0)
    targets = np.concatenate(targets, axis=0)
    digits_all = np.concatenate(digits_all, axis=0)
    return acts, images, targets, digits_all


acts0, images0, targets0, digits0 = collect_vit_activations(model0, test_loader, DEVICE)
print(f'Correctly classified: {len(targets0)}')
for k, v in acts0.items():
    print(f'  {k}: {v.shape}')

In [ ]:
D = EMBED_DIM
W_V  = model0.block.attn.in_proj_weight[2*D:, :].detach().cpu().numpy()  # (32, 32)
W_O  = model0.block.attn.out_proj.weight.detach().cpu().numpy()           # (32, 32)
W_f1 = model0.block.ffn1.weight.detach().cpu().numpy()                    # (64, 32)
W_f2 = model0.block.ffn2.weight.detach().cpu().numpy()                    # (32, 64)

layer_dicts0 = [
    {'type': 'attn', 'name': 'B0-V',
     'weight':       W_V,
     'input_fmap':   acts0['attn_in'],        # (N, 17, 32)
     'attn_weights': acts0['attn_w']},         # (N, 17)
    {'type': 'fc', 'name': 'B0-O',
     'weight':    W_O,
     'input_fmap': acts0['attn_out_cls']},     # (N, 32)
    {'type': 'fc', 'name': 'B0-FFN1',
     'weight':    W_f1,
     'input_fmap': acts0['ffn1_in']},          # (N, 32)
    {'type': 'fc', 'name': 'B0-FFN2',
     'weight':    W_f2,
     'input_fmap': acts0['ffn2_in']},          # (N, 64)
]

print('Layer chain (forward order):')
for d in layer_dicts0:
    print(f"  {d['name']:<10} type={d['type']:<5}  W:{str(d['weight'].shape):<12}  "
          f"input:{d['input_fmap'].shape}")

### 2b — Run BFT

In [ ]:

from src import cached_tree
tree0 = cached_tree('nb04_circuit', lambda: bft(
    layer_dicts0,
    k_max=K_MAX,
    n_branches=N_BRANCHES,
    stimulus_threshold=STIM_THRESHOLD,
    weighting='img_selectivity',
    verbose=1,
    n_jobs=3,
), params=dict(k=K_MAX, b=N_BRANCHES, tau=STIM_THRESHOLD, n=len(targets)))



def get_all_paths(root):
    if not root.children:
        return [[root]]
    return [[root] + p for c in root.children for p in get_all_paths(c)]

paths0 = get_all_paths(tree0.root)
print(f'{len(paths0)} paths from root (B0-FFN2)')
for p in paths0:
    print(f"  path factor-{p[0].factor_idx}  layers: "
          + ' → '.join(n.layer_name for n in p))


### 2c — Exploratory plots: factor overview panels and galleries

In [ ]:
# ── Plot 1+4: Factor overview panels & per-factor galleries (all tree nodes) ──
for node in extract_tree_nodes(tree0):
    layer_name = node.get('layer_name', f'L{node["layer_idx"]}')
    figs1 = plot_factor_overview_panel(node, images0, targets0, CLASS_NAMES,
                                        digit_targets=digits0)
    for k, fig in enumerate(figs1):
        fig.savefig(os.path.join(FIG_DIR, f'factor_overview_{layer_name}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)
    K = node['img_factors'].shape[1]
    for k in range(K):
        fig4 = plot_factor_gallery(node, images0, targets0, CLASS_NAMES, k=k, n=10)
        fig4.savefig(os.path.join(FIG_DIR, f'factor_gallery_{layer_name}_k{k}.pdf'),
                     bbox_inches='tight')
        plt.close(fig4)

print(f'Plot 1+4 saved for all tree nodes.')

### 2d — Exploratory plots: attention spatial grounding

In [ ]:
# ── Plot 2: Attention spatial grounding via plot_input_layer_factors ──────────
def get_attn_nodes(root):
    nodes, queue = [], [root]
    while queue:
        n = queue.pop(0)
        if n.layer_type == 'attn':
            nodes.append(n)
        queue.extend(n.children)
    return nodes

attn_nodes = get_attn_nodes(tree0.root)
print(f'{len(attn_nodes)} attention (B0-V) nodes in tree')

for attn_node in attn_nodes:
    layer_name = attn_node.layer_name
    figs2 = plot_input_layer_factors(attn_node, images0, arch='attn',
                                      image_shape=(1, IMAGE_SIDE, IMAGE_SIDE))
    for k, fig in enumerate(figs2):
        fig.savefig(os.path.join(FIG_DIR, f'input_factors_{layer_name}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)

print(f'Plot 2 saved for {len(attn_nodes)} attention leaf nodes.')


### 2e — Exploratory plots: factor tree per class

In [ ]:
# ── Scaffold graph ─────────────────────────────────────────────────────────────
tree_nodes0 = extract_tree_nodes(tree0)

fig, ax = plt.subplots(figsize=(10, 5))
node_acts = compute_node_activations(tree_nodes0, list(range(len(targets0))))
plot_factor_tree(tree_nodes0, node_acts, ax=ax, title='BFT scaffold (seed 0) — all stimuli')
fig.savefig(os.path.join(FIG_DIR, 'scaffold_all.png'), dpi=120, bbox_inches='tight')
plt.show()

for cl, cname in enumerate(CLASS_NAMES):
    cl_idx = np.where(targets0 == cl)[0]
    fig, ax = plt.subplots(figsize=(10, 5))
    node_acts_cl = compute_node_activations(tree_nodes0, cl_idx)
    plot_factor_tree(tree_nodes0, node_acts_cl, ax=ax, title=f'scaffold — {cname} stimuli')
    fig.savefig(os.path.join(FIG_DIR, f'scaffold_{cname}.png'), dpi=120, bbox_inches='tight')
    plt.show()

## §3 — BFT figures (main paper & appendix)

### 3a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the TinyViT circuits
# Requires: tree0, images0, targets0, digits0  (§2 above)
# Writes:   figures/figdata/nb04_circuits.npz  (+ .json)
# The figures are built in notebooks/fig04_vit.ipynb from this bundle alone. Attn
# nodes also carry their mean CLS attention per factor (attn_mean), which is what
# grounds a factor in image space.
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport

root = tree0.root
CLASSES = list(range(N_CLASSES))                       # even / odd
DIGITS  = list(range(10))

# ── one circuit per root factor, traced down to the first block ────────────
circuits = []
for child in root.children:
    chain, leaf = [root, child], child
    while leaf.children:
        leaf = leaf.children[0]
        chain.append(leaf)
    prof = figexport.class_profile(root, targets0, CLASSES)[int(child.path[0])]
    circuits.append(dict(
        k=int(child.path[0]), profile=prof,
        digit_profile=figexport.class_profile(root, digits0, DIGITS)[int(child.path[0])],
        layer_names=[str(n.layer_name) for n in reversed(chain)],
        layer_types=[str(n.layer_type) for n in reversed(chain)],
        depth=len(chain)))

# ── the whole trace; class profiles for parity *and* digit identity ────────
nodes = figexport.export_tree(root, labels=targets0, classes=CLASSES, images=images0,
                              img_max_side=28, example_max_side=28)
_bfs, _q = [], [root]
while _q:
    _n = _q.pop(0); _bfs.append(_n); _q.extend(_n.children)
for _d, _n in zip(nodes, _bfs):
    _d['digit_profile'] = figexport.class_profile(_n, digits0, DIGITS)

D = figdata.save('nb04_circuits', dict(
    n_classes=N_CLASSES, class_names=[str(CLASS_NAMES[c]) for c in CLASSES],
    digits=DIGITS, image_side=IMAGE_SIDE,
    meta=figexport.trace_meta(tree0, k_max=K_MAX, n_branches=N_BRANCHES,
                              stimulus_threshold=STIM_THRESHOLD,
                              embed_dim=EMBED_DIM, n_heads=N_HEADS, ffn_dim=FFN_DIM),
    circuits=circuits, nodes=nodes,
    stimuli=figexport.example_stimuli(images0, digits0, DIGITS, per_class=8,
                                      max_side=28)))
figdata.summary('nb04_circuits')


### 3b — Appendix figure

In [ ]:
# (appendix figure: build it in notebooks/fig04_vit.ipynb from the exported bundle)


## §4 — Fingerprints

`project_stimuli_onto_tree` re-uses the fixed NMF bases from the seed-0 BFT tree
and projects new stimuli onto them via backward-weighted NNLS.
For the attention layer (B0-V) the input must be supplied as a dict
`{'x_tokens': (N,T,d), 'attn_weights': (N,T)}`.

In [ ]:
# ── Two trees: circuits use the C0 tree (above); fingerprints use a shallower
#    paper-HP tree (nb16 C1.8). Both cached.
from src import cached_tree
tree_circuit = tree0                    # keep the C0 circuit tree for §7/§8
tree0 = cached_tree('nb04_fp', lambda: bft(
    layer_dicts0, k_max=K_MAX_FP, n_branches=N_BRANCHES_FP,
    stimulus_threshold=STIM_THRESHOLD, weighting='img_selectivity', n_jobs=3),
    params=dict(k=K_MAX_FP, b=N_BRANCHES_FP, tau=STIM_THRESHOLD, n=len(targets)))
print('circuit tree:', sum(1 for _ in tree_circuit.nodes()), 'nodes | '
      'fingerprint tree:', sum(1 for _ in tree0.nodes()), 'nodes')

ANALYSIS_CTX = dict(
    tag='nb04', model=model0, tree_circuit=tree_circuit, tree_fp=tree0,
    layer_inputs=[d['input_fmap'] for d in layer_dicts0], labels_task=targets.astype(int),
    labels_fine=targets.astype(int), eval_loader=None,
    label_transform=None, device=DEVICE,
    layer_names=[d['name'] for d in layer_dicts0],
    n_classes=2, k_cap=16, last_extra=2, k_max_cfg=K_MAX,
    skip_pruning='ViT: pruning not wired (attention arbor); see limitations',
    prune_fractions=(0.02, 0.05, 0.1, 0.2), n_random=5, stab_seeds=5)

In [ ]:
# §3 only exports plot data — the paper figures live in
# notebooks/fig04_vit.ipynb — so nothing above changed matplotlib's rcParams.
# Kept so the exploratory plots below render at screen size.
plt.rcParams.update({'figure.dpi': 80})


### 4a — NNLS collection helpers

In [ ]:
def collect_vit_layer_inputs(model, loader, device, only_correct=True, max_per_class=None):
    """Collect layer inputs for new_layer_inputs passed to project_stimuli_onto_tree.

    Returns (new_layer_inputs, images, targets, digits) where new_layer_inputs is
    a list of 4 items in FORWARD order:
      [0] dict {'x_tokens': (N,17,32), 'attn_weights': (N,17)}   — attn layer
      [1] ndarray (N, 32)  — B0-O fc layer
      [2] ndarray (N, 32)  — B0-FFN1
      [3] ndarray (N, 64)  — B0-FFN2
    """
    model.eval()
    acts = {'attn_in': [], 'attn_w': [], 'attn_out': [], 'ffn1_in': [], 'ffn2_in': []}
    images, targets_l, digits_l = [], [], []
    counts = {c: 0 for c in range(N_CLASSES)}

    with torch.no_grad():
        for imgs, digs in loader:
            imgs   = imgs.to(device)
            labels = label_transform_even_odd(digs)
            logits = model(imgs, capture=True)
            preds  = logits.argmax(1).cpu()

            if only_correct:
                mask = (preds == labels)
            else:
                mask = torch.ones(len(labels), dtype=torch.bool)

            if max_per_class is not None:
                for c in range(N_CLASSES):
                    if counts[c] >= max_per_class:
                        mask &= ~(labels == c)

            if not mask.any():
                continue

            blk = model.block
            acts['attn_in'].append(blk._attn_in[mask].cpu().numpy())
            acts['attn_w'].append(blk._attn_w[mask].mean(1)[:, 0, :].cpu().numpy())
            acts['attn_out'].append(blk._attn_out[mask, 0].cpu().numpy())
            acts['ffn1_in'].append(blk._ffn1_in[mask, 0].cpu().numpy())
            acts['ffn2_in'].append(blk._ffn2_in[mask, 0].cpu().numpy())
            images.append(imgs[mask].cpu().numpy())
            targets_l.append(labels[mask].numpy())
            digits_l.append(digs[mask.cpu()].numpy())
            for c in range(N_CLASSES):
                counts[c] += (labels[mask] == c).sum().item()

    for k in acts:
        acts[k] = np.concatenate(acts[k], axis=0)

    new_layer_inputs = [
        {'x_tokens': acts['attn_in'], 'attn_weights': acts['attn_w']},  # idx 0 = B0-V
        acts['attn_out'],    # idx 1 = B0-O
        acts['ffn1_in'],     # idx 2 = B0-FFN1
        acts['ffn2_in'],     # idx 3 = B0-FFN2
    ]
    images    = np.concatenate(images,    axis=0)
    targets_l = np.concatenate(targets_l, axis=0)
    digits_l  = np.concatenate(digits_l,  axis=0)
    return new_layer_inputs, images, targets_l, digits_l


def collect_vit_ood_inputs(model, loader, device, n_max=2000):
    """Collect activations for OOD data (no label filtering)."""
    model.eval()
    acts = {'attn_in': [], 'attn_w': [], 'attn_out': [], 'ffn1_in': [], 'ffn2_in': []}
    images_l = []
    n_seen = 0

    with torch.no_grad():
        for imgs, _ in loader:
            if n_seen >= n_max:
                break
            imgs = imgs.to(device)
            _ = model(imgs, capture=True)
            blk = model.block
            b = min(len(imgs), n_max - n_seen)
            acts['attn_in'].append(blk._attn_in[:b].cpu().numpy())
            acts['attn_w'].append(blk._attn_w[:b].mean(1)[:, 0, :].cpu().numpy())
            acts['attn_out'].append(blk._attn_out[:b, 0].cpu().numpy())
            acts['ffn1_in'].append(blk._ffn1_in[:b, 0].cpu().numpy())
            acts['ffn2_in'].append(blk._ffn2_in[:b, 0].cpu().numpy())
            images_l.append(imgs[:b].cpu().numpy())
            n_seen += b

    for k in acts:
        acts[k] = np.concatenate(acts[k], axis=0)
    new_layer_inputs = [
        {'x_tokens': acts['attn_in'], 'attn_weights': acts['attn_w']},
        acts['attn_out'], acts['ffn1_in'], acts['ffn2_in'],
    ]
    images_l = np.concatenate(images_l, axis=0)
    return new_layer_inputs, images_l


print('NNLS helpers defined.')

### 4b — NNLS round-trip fidelity

In [ ]:
# ── Round-trip test (print Pearson r — no plot) ────────────────────────────────
new_inputs_test, _, _, _ = collect_vit_layer_inputs(model0, test_loader, DEVICE, only_correct=True)
tree_rt = project_stimuli_onto_tree(tree0, new_inputs_test)
orig_f = tree0.root.img_factors
nnls_f = tree_rt.img_factors
N_cmp = min(len(orig_f), len(nnls_f))
K_cmp = min(orig_f.shape[1], nnls_f.shape[1])
corrs = [np.corrcoef(orig_f[:N_cmp, k], nnls_f[:N_cmp, k])[0, 1] for k in range(K_cmp)]
print('Round-trip Pearson r per root factor:')
for k, r in enumerate(corrs):
    print(f'  factor {k}: r={r:.4f}')
print(f'  mean r = {np.mean(corrs):.4f}')


### 4c — Near-OOD: Fashion-MNIST

In [ ]:
# ── Near-OOD: Fashion-MNIST ────────────────────────────────────────────────────
fmnist_loader = DataLoader(
    datasets.FashionMNIST('../data/', train=False, download=True, transform=T.ToTensor()),
    batch_size=64, shuffle=False,
)
FMNIST_CLASSES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']


new_inputs_fmnist, images_fmnist = collect_vit_ood_inputs(model0, fmnist_loader, DEVICE, n_max=2000)

tree_fmnist = project_stimuli_onto_tree(tree0, new_inputs_fmnist)
print(f'Fashion-MNIST OOD: {images_fmnist.shape[0]} samples projected.')

### 4d — Far-OOD: synthetic images

In [ ]:
# ── Far-OOD projections ────────────────────────────────────────────────────────
N_FAR = 500

def make_far_ood(n=N_FAR):
    return {
        'gaussian':    torch.clamp(torch.randn(n, 1, 28, 28) * 0.3 + 0.5, 0, 1),
        'uniform':     torch.zeros(n, 1, 28, 28) + 0.5,
        'checkerboard': torch.tensor(
            np.tile([[0, 1], [1, 0]], (n, 1, 14, 14)).astype(np.float32)),
        'inverted':    1.0 - torch.stack([test_loader.dataset[i][0] for i in range(n)]),
    }

far_ood_tensors = make_far_ood()

far_trees = {}
for ood_name, tensor in far_ood_tensors.items():
    ood_loader = DataLoader(
        TensorDataset(tensor, torch.zeros(N_FAR, dtype=torch.long)),
        batch_size=64,
    )
    nli, imgs_far = collect_vit_ood_inputs(model0, ood_loader, DEVICE, n_max=N_FAR)
    projected = project_stimuli_onto_tree(tree0, nli)
    far_trees[ood_name] = {'projected': projected, 'images': imgs_far, 'nli': nli}
    print(f'{ood_name}: {imgs_far.shape[0]} samples projected')

### 4e — Fingerprint similarity matrix

In [ ]:
# ── Fingerprint similarity matrix ─────────────────────────────────────────────
N_FP = 100

conditions = {
    'even (ID)':    (tree0,        np.where(targets0 == 0)[0][:N_FP]),
    'odd (ID)':     (tree0,        np.where(targets0 == 1)[0][:N_FP]),
    'FashionMNIST': (tree_fmnist,  np.arange(N_FP)),
    'gaussian':     (far_trees['gaussian']['projected'],     np.arange(N_FP)),
    'uniform':      (far_trees['uniform']['projected'],      np.arange(N_FP)),
    'checkerboard': (far_trees['checkerboard']['projected'], np.arange(N_FP)),
    'inverted':     (far_trees['inverted']['projected'],     np.arange(N_FP)),
}

fp_matrices = {cname: extract_fingerprint_matrix(tree_c, idx_c)
               for cname, (tree_c, idx_c) in conditions.items()}

cond_names = list(conditions.keys())
n_conds    = len(cond_names)
sim_matrix = np.zeros((n_conds, n_conds))
for i, ci in enumerate(cond_names):
    for j, cj in enumerate(cond_names):
        S = cosine_similarity(fp_matrices[ci], fp_matrices[cj])
        sim_matrix[i, j] = S.mean()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(n_conds)); ax.set_xticklabels(cond_names, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(n_conds)); ax.set_yticklabels(cond_names, fontsize=9)
plt.colorbar(im, ax=ax, label='mean cosine similarity')
for i in range(n_conds):
    for j in range(n_conds):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                color='white' if sim_matrix[i, j] < 0.5 else 'black', fontsize=7)
ax.set_title('BFT fingerprint similarity matrix', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fingerprint_sim_matrix.pdf'), dpi=120, bbox_inches='tight')
plt.show()

### 4f — Fingerprint embeddings and intra/inter-class similarity

In [ ]:
# ── Plot 7: embedding comparison ──────────────────────────────────────────────
N_EACH = 100; rng_m = np.random.default_rng(7)

F_parts, la_parts, full_la_parts, emb_labels, emb_digits, emb_conds = [], [], [], [], [], []

# ID MNIST even/odd
id_even_idx = np.where(targets0 == 0)[0][:N_FP]
id_odd_idx  = np.where(targets0 == 1)[0][:N_FP]
id_all_idx  = np.concatenate([id_even_idx, id_odd_idx])
F_parts.append(np.concatenate([fp_matrices['even (ID)'], fp_matrices['odd (ID)']], axis=0))
la_parts.append(np.vstack([
    np.array([acts0['ffn2_in'][i] for i in id_even_idx]),
    np.array([acts0['ffn2_in'][i] for i in id_odd_idx]),
]))
full_la_parts.append(np.concatenate([
    acts0['attn_in'][id_all_idx].mean(1),
    acts0['attn_out_cls'][id_all_idx],
    acts0['ffn1_in'][id_all_idx],
    acts0['ffn2_in'][id_all_idx],
], axis=1))
emb_labels.extend(targets0[id_even_idx].tolist() + targets0[id_odd_idx].tolist())
emb_digits.extend(digits0[id_even_idx].tolist()  + digits0[id_odd_idx].tolist())
emb_conds.extend(['ID'] * (2 * N_FP))

# Near-OOD Fashion-MNIST
F_parts.append(fp_matrices['FashionMNIST'])
la_parts.append(np.array([new_inputs_fmnist[-1][i] for i in range(N_FP)]))
full_la_parts.append(np.concatenate([
    new_inputs_fmnist[0]['x_tokens'][:N_FP].mean(1),
    new_inputs_fmnist[1][:N_FP],
    new_inputs_fmnist[2][:N_FP],
    new_inputs_fmnist[3][:N_FP],
], axis=1))
emb_labels.extend([2] * N_FP)
emb_digits.extend([-1] * N_FP)
emb_conds.extend(['FashionMNIST'] * N_FP)

# Far-OOD (first 2 types)
for ood_name in list(far_trees.keys())[:2]:
    d = far_trees[ood_name]
    nli = d['nli']
    n = min(N_FP, len(nli[-1]))
    F_parts.append(fp_matrices[ood_name])
    la_parts.append(np.array([nli[-1][i] for i in range(n)]))
    full_la_parts.append(np.concatenate([
        nli[0]['x_tokens'][:n].mean(1),
        nli[1][:n],
        nli[2][:n],
        nli[3][:n],
    ], axis=1))
    emb_labels.extend([3] * n)
    emb_digits.extend([-1] * n)
    emb_conds.extend([ood_name] * n)

F_joint = np.concatenate(F_parts, axis=0)
la_joint = np.vstack(la_parts)
emb_labels = np.array(emb_labels)
emb_digits = np.array(emb_digits)

full_class_names = {0: 'even', 1: 'odd', 2: 'FashionMNIST', 3: 'far-OOD'}
fig7 = plot_embedding_comparison(
    F_joint, la_joint, emb_labels, full_class_names,
    condition_labels=emb_conds,
    far_ood_conditions=list(far_trees.keys())[:2],
    activations_all=np.concatenate(full_la_parts, axis=0),
    title='MNIST even/odd — ID / Near-OOD / Far-OOD')
fig7.savefig(os.path.join(FIG_DIR, 'embedding_comparison.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig7)

# Intra vs inter-class similarity histogram
F_id = np.concatenate([fp_matrices['even (ID)'], fp_matrices['odd (ID)']], axis=0)
lbl_id = np.array([0] * N_FP + [1] * N_FP)
S_id = cosine_similarity(F_id)
intra_vals, inter_vals = [], []
for ci in range(N_CLASSES):
    mask = lbl_id == ci
    intra = S_id[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for cj in range(ci + 1, N_CLASSES):
        inter_vals.extend(S_id[np.ix_(mask, lbl_id == cj)].ravel())
intra_arr, inter_arr = np.array(intra_vals), np.array(inter_vals)
print(f'Intra-class: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')
fig_ii, ax = plt.subplots(figsize=(6, 4))
ax.hist(intra_arr, bins=50, alpha=0.6, label='Intra-class', density=True)
ax.hist(inter_arr, bins=50, alpha=0.6, label='Inter-class', density=True)
ax.axvline(intra_arr.mean(), color='C0', ls='--'); ax.axvline(inter_arr.mean(), color='C1', ls='--')
ax.set(xlabel='Cosine similarity', ylabel='Density',
       title='Factor fingerprint: intra vs inter-class')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fingerprint_intra_inter.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_ii)

## §5 — Fingerprint figures (main paper & appendix)

### 5a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the TinyViT fingerprints
# Requires: tree0, targets0, digits0  (§2), corrs (§4b), tree_fmnist (§4c),
#           far_trees (§4d), fp_matrices / cond_names / sim_matrix (§4e)
# Writes:   figures/figdata/nb04_fingerprints.npz  (+ .json)
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport
from src.paper_figures import unit

CLASSES = list(range(N_CLASSES))
DIGITS  = list(range(10))

F  = extract_fingerprint_matrix(tree0, np.arange(len(targets0)))
F_fm = extract_fingerprint_matrix(tree_fmnist, np.arange(N_FP))
F_far = {name: extract_fingerprint_matrix(d['projected'], np.arange(N_FP))
         for name, d in far_trees.items()}

_dims, _q = [], [tree0.root]
while _q:
    _n = _q.pop(0)
    for _k in range(_n.img_factors.shape[1]):
        _dims.append((_n.layer_idx, _n.path[0] if _n.path else -1, _k))
    _q.extend(_n.children)
_dims = np.array(_dims)
_top  = _dims[:, 0].max()
BLOCKS = [[int(r)] + [i for i in range(len(_dims))
                      if _dims[i, 0] != _top and _dims[i, 1] == _dims[r, 2]]
          for r in np.where(_dims[:, 0] == _top)[0]]


def centroid_cos(Fm):
    U = unit(Fm)
    c = U.mean(0)
    return U @ (c / (np.linalg.norm(c) + 1e-12))


CENT = unit(np.stack([F[targets0 == c].mean(0) for c in CLASSES]))
COND = ([dict(label='ID test', values=centroid_cos(F), color_key='id_data'),
         dict(label='Fashion-MNIST', values=centroid_cos(F_fm), color_key='near_ood')] +
        [dict(label=n, values=centroid_cos(F_far[n]), color_key='far_ood')
         for n in far_trees])
LIKE = [dict(label=lab, values=(unit(X) @ CENT.T).max(1), color_key=ck)
        for lab, X, ck in ([('ID test', F, 'id_data'),
                            ('Fashion-MNIST', F_fm, 'near_ood')] +
                           [(n, F_far[n], 'far_ood') for n in far_trees])]

_sub = figexport.subsample_by_class(digits0, DIGITS, 150, seed=0)

D = figdata.save('nb04_fingerprints', dict(
    n_classes=N_CLASSES, n_factors=F.shape[1], dims=_dims,
    col_order=[c for b in BLOCKS for c in b],
    blk_edge=np.cumsum([len(b) for b in BLOCKS])[:-1],
    block_sizes=[len(b) for b in BLOCKS],
    fp_mean_by_class=np.stack([F[targets0 == c].mean(0) for c in CLASSES]),
    fp_mean_by_digit=np.stack([F[digits0 == d].mean(0) for d in DIGITS]),
    cond=COND, like=LIKE,
    roundtrip_corrs=np.asarray(corrs, float),
    condition_similarity=dict(names=[str(c) for c in cond_names],
                              matrix=np.asarray(sim_matrix, np.float32)),
    fp=dict(id=F[_sub].astype(np.float32), id_targets=np.asarray(targets0)[_sub],
            id_digits=np.asarray(digits0)[_sub], id_index=_sub,
            fmnist=F_fm.astype(np.float32),
            far=[dict(label=n, F=F_far[n].astype(np.float32)) for n in far_trees])))
figdata.summary('nb04_fingerprints')


### 5b — Appendix figure

In [ ]:
# (appendix figure: build it in notebooks/fig04_vit.ipynb from the exported bundle)


## §6 — Hyperparameter check (held-out arbor R²)

Re-derives the per-layer circuit rank with the metric-free C0 rule (`src.hp_selection`), on the circuit tree's own arbors. Confirms the `K_MAX` in §1 sits at the reconstruction plateau; reads no fingerprint metric. Cached.

In [ ]:
from src import node_pos_arbor, nodes_per_layer, select_ranks, cached_result
_C = ANALYSIS_CTX
_npl = nodes_per_layer(_C['tree_circuit'], max_nodes=2)
_arbors = {li: [node_pos_arbor(nd, _C['layer_inputs'][li]) for nd in nds]
           for li, nds in _npl.items()}
hp_sel = cached_result(
    _C['tag'] + '_hpsel',
    lambda: select_ranks(_arbors, _C['labels_task'], k_cap=_C['k_cap'],
                         n_classes=_C['n_classes'], last_extra=_C['last_extra']),
    params=dict(kcap=_C['k_cap'], n=len(_C['labels_task']),
                kmax=list(_C['tree_circuit'].root.lambdas.shape)))
print('held-out K* per layer:', hp_sel['profile']['k_from_criterion'])
print('assembled profile     k_max=%s  n_branches=%s'
      % (hp_sel['profile']['k_max'], hp_sel['profile']['n_branches']))
print('§1 circuit k_max was :', _C.get('k_max_cfg'))

## §7 — Validation (faithfulness + class-relevant structure)

On the **circuit** tree: NMF init-stability per layer, causal-reconstruction fidelity (fc layers only), and the weight-term control (arbor-NMF vs activation-NMF separability). All via `src`; cached.

In [ ]:
from src import (compute_nmf_stability, summarize_validation, node_pos_arbor,
                 nodes_by_layer, weight_term_control, cached_result)
import numpy as _np
_C = ANALYSIS_CTX
def _run_validation():
    tc = _C['tree_circuit']
    nbl = nodes_by_layer(tc)
    stability = {}
    for li, nd in nbl.items():
        X = node_pos_arbor(nd, _C['layer_inputs'][li])
        k = int(nd.img_factors.shape[1])
        sim, _ = compute_nmf_stability(
            X if X.shape[0] <= 800 else X[_np.random.default_rng(0).choice(X.shape[0], 800, False)],
            k, n_seeds=_C.get('stab_seeds', 5), max_iter=300)
        stability[int(li)] = float(sim[~_np.eye(len(sim), dtype=bool)].mean())
    recon = summarize_validation(tc.nodes())        # None when no fc node was validated
    wtc = weight_term_control(tc, _C['labels_task'], _C['layer_inputs'])
    return {'stability': stability,
            'recon_overall': (recon['overall'] if recon else None),
            'weight_term': wtc}
val = cached_result(_C['tag'] + '_validation', _run_validation,
                    params=dict(n=len(_C['labels_task']), tag='circ'))
print('NMF stability per layer:', {k: round(v, 3) for k, v in val['stability'].items()})
if val['recon_overall']:
    print('causal recon preact_R2 (median):',
          round(val['recon_overall']['preact_r2']['median'], 3))
print('weight-term control: arbor NMF sil=%.3f  vs activation NMF sil=%.3f'
      % (val['weight_term']['arbor_nmf']['sil'], val['weight_term']['activation_nmf']['sil']))

## §8 — Causal pruning

Prunes each class circuit's weights in BFT-importance order on the **circuit** tree and measures target vs bystander accuracy (`src.pruning`, wrapping `ablation_sweep`). One seed here; add checkpoints for the full seed×class grid on the cluster. Cached.

In [ ]:
from src import run_pruning, cached_result
import numpy as _np
_C = ANALYSIS_CTX
if _C.get('skip_pruning'):
    print('pruning skipped for this model:', _C.get('skip_pruning'))
    prune = None
else:
    _reps = [{'seed': 0, 'model': _C['model'], 'tree': _C['tree_circuit'],
              'layer_names': _C.get('layer_names')}]
    _targets = (list(range(_C['n_classes'])) if _C['n_classes'] <= 10
                else list(range(_C['n_classes']))[:10])
    prune = cached_result(
        _C['tag'] + '_pruning',
        lambda: run_pruning(_reps, _C['eval_loader'], _targets, n_classes=_C['n_classes'],
                            fractions=_C.get('prune_fractions', (0.02, 0.05, 0.1, 0.2)),
                            label_transform=_C.get('label_transform'), device=_C.get('device'),
                            n_random_repeats=_C.get('n_random', 5), verbose=1)['aggregate'],
        params=dict(n=len(_targets), tag='prune'))
    for m in ('bft_top', 'bft_bottom', 'random'):
        if m in prune['methods']:
            print('%-11s target drop@0.2 (mean over classes): %+.3f'
                  % (m, _np.mean(prune['drops'][m]['target'])))

## §9 — Fingerprint separability (C1.8)

On the **fingerprint** tree: is the factor fingerprint more class-separable than the network's own activations, and where in the tree does that live? `src.separability` gives silhouette + kNN for the whole tree, its upper/lower slices, and the penultimate / full-activation baselines (native and dim-matched). Cached.

In [ ]:
from src import separability_evaluate, cached_result
_C = ANALYSIS_CTX
sep = cached_result(
    _C['tag'] + '_separability',
    lambda: separability_evaluate(_C['tree_fp'], _C['labels_fine'], _C['layer_inputs']),
    params=dict(n=len(_C['labels_fine']), tag='fp'))
_n = sep['native']
print('native silhouette:  fp_full=%.3f  output_only=%.3f  top_half=%.3f  spine=%.3f'
      % (_n['fp_full']['sil'], _n.get('fp_output_only', {}).get('sil', float('nan')),
         _n.get('fp_top_half', {}).get('sil', float('nan')), _n.get('fp_spine', {}).get('sil', float('nan'))))
print('activation baselines: penult=%.3f  full=%.3f'
      % (_n['act_penult']['sil'], _n['act_full']['sil']))
_p = sep['paired'].get('fp_full__vs__act_penult')
if _p:
    print('dim-matched @%d: fp(pca)=%.3f vs penult(pca)=%.3f'
          % (_p['match_dim'], _p['A_pca']['sil'], _p['B_pca']['sil']))